# Event Detection using SmolVLM2

Built upon the demo notebook (`Demo_Event_Summarization_using_VLMs.ipynb`) for Assignment 2.

Given a video, this notebook uses **SmolVLM2-2.2B-Instruct** (same model as the demo) to detect salient events in two modes:

1. **Event-only output** — a numbered list of salient events  
   Example: `Salient event 1: person enters the room`

2. **Event + timestamp output** — events with approximate temporal boundaries  
   Example: `Salient event 1: person enters the room, 00:03 - 00:07`

The notebook follows the same inference pipeline as the demo and extends it with improved prompt design, output parsing, deduplication, and file saving.

## Installation

In [3]:
!pip install num2words av pandas transformers torch torchvision

Defaulting to user installation because normal site-packages is not writeable


## Model Loading

We use the same model as the demo: **SmolVLM2-2.2B-Instruct**.
It is a lightweight vision-language model designed for efficient video understanding.

In [5]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import cv2
from PIL import Image
import random
import json
import re
import os
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher

model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
).to("cuda")

print(f"Model loaded: {model_path}")
print(f"Device: {next(model.parameters()).device}")

Loading checkpoint shards:  50%|█████     | 1/2 [00:17<00:17, 17.83s/it]


KeyboardInterrupt: 

## Video Input

Set the path to your video file below. The notebook supports common formats: `.mp4`, `.avi`, `.mov`, etc.

In [ ]:
# -------------------------------------------------------
# Set your video path here
video_path = "video_22.mp4"
# -------------------------------------------------------

video_name = Path(video_path).stem
print(f"Video path : {video_path}")
print(f"Video name : {video_name}")

## Frame Extraction

We reuse and extend the frame sampling logic from the demo notebook:

- **`sample_frames`** — random sampling from the demo (kept unchanged for reference)
- **`sample_frames_uniform`** — uniform temporal sampling (1 frame every N seconds), used for timestamp-aware inference so we always know exactly when each frame was captured

Uniform sampling is critical for the timestamp output because it lets us tell the model at what time each frame appeared.

In [ ]:
# ── Utility ──────────────────────────────────────────────────────────────────

def format_timestamp(seconds):
    """Convert seconds (float) to MM:SS string."""
    m = int(seconds) // 60
    s = int(seconds) % 60
    return f"{m:02d}:{s:02d}"


def get_video_info(video_path):
    """Return basic metadata: fps, total_frames, duration (seconds)."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0
    cap.release()
    return {"fps": fps, "total_frames": total_frames, "duration": duration}


# ── Random sampling (from demo notebook — kept unchanged) ─────────────────────

def sample_frames(video_path, num_frames):
    """Random frame sampling (original demo logic)."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        raise ValueError("Video has no frames.")
    frame_indices = sorted(random.sample(range(total_frames), min(num_frames, total_frames)))
    frames = []
    current_idx = 0
    target_idx_set = set(frame_indices)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if current_idx in target_idx_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame))
        current_idx += 1
    cap.release()
    return frames


# ── Uniform temporal sampling (new — for timestamp-aware inference) ────────────

def sample_frames_uniform(video_path, interval_seconds=2.0, max_frames=32):
    """
    Sample frames at a fixed time interval.
    Returns:
        frames     — list of PIL Images
        timestamps — list of float (seconds)
    Caps at max_frames to avoid memory issues on long videos.
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        raise ValueError("Could not read video FPS.")

    frame_interval = max(1, int(fps * interval_seconds))
    frames, timestamps = [], []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
            timestamps.append(round(frame_idx / fps, 2))
        frame_idx += 1

    cap.release()

    # Subsample if too many frames
    if len(frames) > max_frames:
        indices = [int(i * len(frames) / max_frames) for i in range(max_frames)]
        frames = [frames[i] for i in indices]
        timestamps = [timestamps[i] for i in indices]

    return frames, timestamps


# ── Inspect the video ─────────────────────────────────────────────────────────

info = get_video_info(video_path)
print(f"FPS          : {info['fps']:.1f}")
print(f"Duration     : {format_timestamp(info['duration'])}  ({info['duration']:.1f}s)")
print(f"Total frames : {info['total_frames']}")

frames_uniform, timestamps_uniform = sample_frames_uniform(video_path, interval_seconds=2.0)
print(f"Uniform sample: {len(frames_uniform)} frames  "
      f"({timestamps_uniform[0]:.1f}s … {timestamps_uniform[-1]:.1f}s)")

## Prompt Design

Good prompts are the most important part of VLM-based event detection.

Both prompts below instruct the model to:
- Focus on **salient events**, not per-frame pixel descriptions
- **Group** similar or continuous actions into a single event (no repetition)
- Output events in **chronological order**
- Use a **strict numbered format** (`Salient event N: ...`) to make parsing reliable
- Scale the number of events dynamically based on video duration

**Prompt A** — event-only (uses the video path directly, as in the demo)  
**Prompt B** — event + timestamps (injects frame timestamps into the prompt so the model can anchor events in time)

In [ ]:
def _expected_n_events(duration_seconds):
    """Estimate a reasonable number of events for this video length."""
    return max(3, min(10, int(duration_seconds / 10)))


def build_event_only_prompt(duration_seconds):
    """Prompt A: return salient events without timestamps."""
    n = _expected_n_events(duration_seconds)
    return (
        "You are a video analyst. You are given a sequence of frames sampled from a video.\n\n"
        f"Task: identify the {n} most salient events in this video.\n\n"
        "Rules:\n"
        "- Focus on significant actions or scene changes, not background details.\n"
        "- Group similar or continuous actions into ONE event — do not repeat the same action.\n"
        "- List events in chronological order.\n"
        "- One short sentence per event; be specific about who does what.\n"
        "- Do NOT output anything besides the numbered list.\n\n"
        "Output format — follow EXACTLY, no extra lines:\n"
        "Salient event 1: [description]\n"
        "Salient event 2: [description]\n"
        "..."
    )


def build_timestamp_prompt(duration_seconds, timestamps):
    """Prompt B: return salient events with MM:SS - MM:SS timestamps."""
    n = _expected_n_events(duration_seconds)
    ts_labels = ", ".join(format_timestamp(t) for t in timestamps)
    end_ts = format_timestamp(duration_seconds)
    return (
        "You are a video analyst. You are given frames from a video "
        f"(total duration: {end_ts}).\n"
        f"The frames were captured at these times: {ts_labels}.\n\n"
        f"Task: identify the {n} most salient events and estimate their start and end timestamps.\n\n"
        "Rules:\n"
        "- Focus on significant actions or scene changes, not background details.\n"
        "- Group similar or continuous actions into ONE event — do not repeat the same action.\n"
        "- List events in chronological order.\n"
        "- One short sentence per event; be specific about who does what.\n"
        f"- Timestamps must be in MM:SS format and stay within 00:00 – {end_ts}.\n"
        "- Do NOT output anything besides the numbered list.\n\n"
        "Output format — follow EXACTLY, no extra lines:\n"
        "Salient event 1: [description], MM:SS - MM:SS\n"
        "Salient event 2: [description], MM:SS - MM:SS\n"
        "..."
    )


# Preview both prompts
event_only_prompt = build_event_only_prompt(info["duration"])
timestamp_prompt = build_timestamp_prompt(info["duration"], timestamps_uniform)

print("=" * 60)
print("PROMPT A — event only")
print("=" * 60)
print(event_only_prompt)
print()
print("=" * 60)
print("PROMPT B — event + timestamps")
print("=" * 60)
print(timestamp_prompt)

## VLM Inference Functions

We define two inference functions, each following the same pattern as the demo notebook:

| Function | Frame strategy | Prompt |
|---|---|---|
| `run_vlm_event_only` | Pass video path directly (`num_frames=16`) | Prompt A |
| `run_vlm_event_with_timestamps` | Uniform frame sample + timestamp labels | Prompt B |

Both functions use `processor.apply_chat_template` and `model.generate` exactly as in the demo.

In [ ]:
def _extract_assistant_reply(full_text):
    """Strip the chat history from the generated text; keep only the model reply."""
    if "Assistant:" in full_text:
        return full_text.split("Assistant:")[-1].strip()
    return full_text.strip()


def run_vlm_event_only(video_path, num_frames=16):
    """
    Event-only inference.
    Uses the direct video-path approach from the demo notebook.
    Returns (output_text, prompt_used).
    """
    info = get_video_info(video_path)
    prompt = build_event_only_prompt(info["duration"])

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "video", "path": video_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        num_frames=num_frames,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=256)
    generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

    return _extract_assistant_reply(generated_texts[0]), prompt


def run_vlm_event_with_timestamps(video_path, interval_seconds=2.0):
    """
    Event + timestamp inference.
    Uses uniformly sampled frames with timestamp labels injected into the message
    so the model can anchor events to specific times.
    Returns (output_text, prompt_used).
    """
    info = get_video_info(video_path)
    frames, timestamps = sample_frames_uniform(video_path, interval_seconds=interval_seconds)
    prompt = build_timestamp_prompt(info["duration"], timestamps)

    # Build content: interleave timestamp labels and frames, then the prompt
    content = []
    for frame, ts in zip(frames, timestamps):
        content.append({"type": "text", "text": f"[Frame at {format_timestamp(ts)}]"})
        content.append({"type": "image", "url": frame})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=256)
    generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

    return _extract_assistant_reply(generated_texts[0]), prompt

## Run Inference

Run both functions on the input video. This is the main computation step.

In [ ]:
print("Running event-only inference …")
event_only_output, event_only_prompt = run_vlm_event_only(video_path)

print("\n" + "=" * 60)
print("EVENT-ONLY OUTPUT")
print("=" * 60)
print(event_only_output)

In [ ]:
print("Running event + timestamp inference …")
timestamp_output, timestamp_prompt = run_vlm_event_with_timestamps(video_path)

print("\n" + "=" * 60)
print("EVENT + TIMESTAMP OUTPUT")
print("=" * 60)
print(timestamp_output)

## Save Outputs to Files

We save all results in three formats:

| File | Contents |
|---|---|
| `event_only_[video].txt` | Plain-text event-only output |
| `event_timestamp_[video].txt` | Plain-text event + timestamp output |
| `vlm_outputs_[video].json` | Full record: paths, model name, prompts, and outputs |

In [ ]:
def save_outputs(video_path, model_name,
                 event_only_prompt, event_only_output,
                 timestamp_prompt, timestamp_output):
    """Save results to .txt and .json files."""
    name = Path(video_path).stem

    txt_event_only = f"event_only_{name}.txt"
    txt_timestamp  = f"event_timestamp_{name}.txt"
    json_path      = f"vlm_outputs_{name}.json"

    with open(txt_event_only, "w") as f:
        f.write(event_only_output)

    with open(txt_timestamp, "w") as f:
        f.write(timestamp_output)

    record = {
        "video_path":         video_path,
        "model_name":         model_name,
        "event_only_prompt":  event_only_prompt,
        "event_only_output":  event_only_output,
        "timestamp_prompt":   timestamp_prompt,
        "timestamp_output":   timestamp_output,
    }
    with open(json_path, "w") as f:
        json.dump(record, f, indent=2)

    print(f"Saved → {txt_event_only}")
    print(f"Saved → {txt_timestamp}")
    print(f"Saved → {json_path}")


save_outputs(
    video_path=video_path,
    model_name=model_path,
    event_only_prompt=event_only_prompt,
    event_only_output=event_only_output,
    timestamp_prompt=timestamp_prompt,
    timestamp_output=timestamp_output,
)

## Output Parser

We parse the raw model text into structured DataFrames:

- **`event_only_df`** — columns: `event_number`, `description`
- **`timestamp_df`** — columns: `event_number`, `description`, `start_time`, `end_time`

The regex patterns match the strict output format defined in the prompts.

In [ ]:
def parse_event_only(text):
    """Parse event-only output into a DataFrame."""
    pattern = r"Salient event (\d+):\s*(.+)"
    matches = re.findall(pattern, text, re.IGNORECASE)
    rows = [
        {"event_number": int(n), "description": desc.strip()}
        for n, desc in matches
    ]
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["event_number", "description"])


def parse_timestamp_events(text):
    """Parse event + timestamp output into a DataFrame."""
    # Match: Salient event N: description, MM:SS - MM:SS
    pattern = r"Salient event (\d+):\s*(.+?),\s*(\d{1,2}:\d{2})\s*-\s*(\d{1,2}:\d{2})"
    matches = re.findall(pattern, text, re.IGNORECASE)
    rows = [
        {
            "event_number": int(n),
            "description":  desc.strip(),
            "start_time":   start.strip(),
            "end_time":     end.strip(),
        }
        for n, desc, start, end in matches
    ]
    cols = ["event_number", "description", "start_time", "end_time"]
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=cols)


event_only_df   = parse_event_only(event_only_output)
timestamp_df    = parse_timestamp_events(timestamp_output)

print("=" * 50)
print(f"Event-only DataFrame  ({len(event_only_df)} events)")
print("=" * 50)
display(event_only_df)

print()
print("=" * 50)
print(f"Timestamp DataFrame  ({len(timestamp_df)} events)")
print("=" * 50)
display(timestamp_df)

## Output Cleaning — Remove Duplicate Events

VLMs sometimes generate near-duplicate events (e.g., the same action described slightly differently in consecutive events). We remove these using a string-similarity threshold:

- If two event descriptions are more than **70% similar**, the later one is dropped.
- Event numbers are renumbered after cleaning.

In [ ]:
def _similarity(a, b):
    """String similarity ratio (0–1) between two descriptions."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def remove_duplicate_events(df, threshold=0.70):
    """
    Remove duplicate or near-duplicate events from a DataFrame.
    Two events are considered duplicates if their description similarity > threshold.
    Keeps the first occurrence; renumbers events afterwards.
    """
    if df.empty or "description" not in df.columns:
        return df.copy()

    kept_rows = []
    kept_descs = []

    for _, row in df.iterrows():
        desc = row["description"]
        is_duplicate = any(_similarity(desc, prev) > threshold for prev in kept_descs)
        if not is_duplicate:
            kept_rows.append(row)
            kept_descs.append(desc)

    cleaned = pd.DataFrame(kept_rows).reset_index(drop=True)
    if not cleaned.empty:
        cleaned["event_number"] = range(1, len(cleaned) + 1)
    return cleaned


event_only_df_clean = remove_duplicate_events(event_only_df)
timestamp_df_clean  = remove_duplicate_events(timestamp_df)

print(f"Event-only : {len(event_only_df)} → {len(event_only_df_clean)} events after deduplication")
print()
print("=" * 50)
print("Cleaned Event-only DataFrame")
print("=" * 50)
display(event_only_df_clean)

print()
print(f"Timestamp  : {len(timestamp_df)} → {len(timestamp_df_clean)} events after deduplication")
print()
print("=" * 50)
print("Cleaned Timestamp DataFrame")
print("=" * 50)
display(timestamp_df_clean)

## Final Outputs — Formatted Display

Print the final results in the exact required format.

In [ ]:
print("=" * 60)
print("FINAL: Event-only output")
print("=" * 60)
for _, row in event_only_df_clean.iterrows():
    print(f"Salient event {row['event_number']}: {row['description']}")

print()
print("=" * 60)
print("FINAL: Event + timestamp output")
print("=" * 60)
for _, row in timestamp_df_clean.iterrows():
    print(f"Salient event {row['event_number']}: {row['description']}, "
          f"{row['start_time']} - {row['end_time']}")

## Summary

This notebook:

1. **Loads SmolVLM2-2.2B-Instruct** — the same model as the demo notebook
2. **Extracts frames** from the input video using uniform temporal sampling (for timestamps) and the demo's direct video-path approach (for event-only)
3. **Runs two inference passes** with carefully designed prompts that prevent repetition and enforce strict output formats
4. **Saves outputs** to `.txt` and `.json` files
5. **Parses outputs** into structured DataFrames
6. **Removes duplicate events** using string-similarity deduplication
7. **Prints final results** in the exact required format

### Output files produced
| File | Description |
|---|---|
| `event_only_[video].txt` | Event-only model output |
| `event_timestamp_[video].txt` | Event + timestamp model output |
| `vlm_outputs_[video].json` | Full JSON record (prompts + outputs) |